# Reliability and attack-success intervals (Module 16)

> **Scope.** Run this only against models and apps you own or the toy targets defined here. No third-party systems, no real people's data. Everything in this repository is synthetic.

Upload the CSV exported from the hosted measurement lab (`m16-outcomes.csv`) to this runtime, or run the cell below to use a small synthetic example.

In [ ]:
import csv, io, math, random, collections, os

if os.path.exists("m16-outcomes.csv"):
    rows = list(csv.DictReader(open("m16-outcomes.csv")))
else:
    rng = random.Random(0)
    rows = [{"item_id": f"I{i:02d}", "behaviour": f"B{(i - 1) // 4 + 1}", "run": str(r), "success": str(int(rng.random() < 0.1 + 0.02 * (i % 7)))}
            for i in range(1, 41) for r in range(1, 6)]
print(len(rows), "outcomes")

In [ ]:
def wilson(x, n, z=1.96):
    p = x / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def cohens_kappa(a, b):
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    pa1, pb1 = sum(a) / n, sum(b) / n
    pe = pa1 * pb1 + (1 - pa1) * (1 - pb1)
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")

def cluster_bootstrap(clusters, B=2000, seed=0):
    rng = random.Random(seed)
    stats = []
    for _ in range(B):
        sample = [rng.choice(clusters) for _ in clusters]
        stats.append(sum(map(sum, sample)) / sum(map(len, sample)))
    stats.sort()
    return stats[int(0.025 * B)], stats[int(0.975 * B)]

run1 = [int(r["success"]) for r in rows if r["run"] == "1"]
print("run-1 ASR", sum(run1) / len(run1), "Wilson", wilson(sum(run1), len(run1)))
by_item = collections.defaultdict(dict)
for r in rows:
    by_item[r["item_id"]][int(r["run"])] = int(r["success"])
pairs = [(v[1], v[2]) for v in by_item.values() if 1 in v and 2 in v]
print("kappa run1 vs run2", cohens_kappa([a for a, _ in pairs], [b for _, b in pairs]))
clusters = collections.defaultdict(list)
for r in rows:
    clusters[r["behaviour"]].append(int(r["success"]))
print("cluster bootstrap 95%", cluster_bootstrap(list(clusters.values())))